In [1]:
import pandas as pd

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
ratings = pd.read_csv('u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [45]:
movies = pd.read_csv('u.item', sep='|',encoding='latin-1',header=None)
movies = movies[[0, 1]]
movies.columns = ['item_id', 'title']
df = pd.merge(ratings, movies, on='item_id')
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df

,user_id,item_id,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)
...,...,...,...,...,...
99995,880,476,3,1997-11-22 05:10:44,"First Wives Club, The (1996)"
99996,716,204,5,1997-11-17 19:39:03,Back to the Future (1985)
99997,276,1090,1,1997-09-20 22:49:55,Sliver (1993)
99998,13,225,2,1997-12-17 22:52:36,101 Dalmatians (1996)


In [33]:
ratings = pd.read_csv('u.data', sep='\t',
                      names=['user_id', 'item_id', 'rating', 'timestamp'])

movies = pd.read_csv('u.item', sep='|',
                     encoding='latin-1', header=None)

movies = movies[[0, 1]]
movies.columns = ['item_id', 'title']

df = pd.merge(ratings, movies, on='item_id')

df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

df.head()

,user_id,item_id,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)


In [46]:
df.shape

(100000, 5)

In [47]:
df["item_id"].nunique()

1682

In [48]:
df["user_id"].nunique()

943

In [49]:
df.isna().sum()

user_id      0
item_id      0
rating       0
timestamp    0
title        0
dtype: int64

### 1. Building Popularity

In [53]:
def popular_recommendations(top_n=5):
    pop = df.groupby('title').agg({'rating': ['mean', 'count']})
    pop.columns = ['avg_rating', 'count']
    pop = pop[pop['count'] > 50]  
    pop = pop.sort_values(by=['avg_rating'], ascending=False)
    return pop.head(top_n).index.tolist()

In [54]:
popular_recommendations()

['Close Shave, A (1995)',
 "Schindler's List (1993)",
 'Wrong Trousers, The (1993)',
 'Casablanca (1942)',
 'Wallace & Gromit: The Best of Aardman Animation (1996)']

In [22]:
def trending_recommendations(top_n=5):
    recent = df[df['timestamp'] > '1998-01-01']
    trend = recent.groupby('title').agg({'rating': ['mean', 'count']})
    trend.columns = ['avg_rating', 'count']
    trend = trend.sort_values(by=['count'], ascending=False)
        return trend.head(top_n).index.tolist()

In [25]:
cv = CountVectorizer(stop_words='english')
matrix = cv.fit_transform(movies['title'])
content_similarity = cosine_similarity(matrix)

In [26]:
def content_recommendations(movie_name, top_n=5):
    if movie_name not in movies['title'].values:
        return ["Movie not found"]

    idx = movies[movies['title'] == movie_name].index[0]

    scores = list(enumerate(content_similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    return [movies.iloc[i[0]].title for i in scores]

In [27]:
user_item = df.pivot_table(index='user_id', columns='title', values='rating').fillna(0)

collab_similarity = cosine_similarity(user_item.T)

collab_df = pd.DataFrame(collab_similarity,
                         index=user_item.columns,
                         columns=user_item.columns)

In [28]:
def collaborative_recommendations(movie_name, top_n=5):
    if movie_name not in collab_df.columns:
        return ["Movie not found"]

    scores = collab_df[movie_name].sort_values(ascending=False)[1:top_n+1]

    return scores.index.tolist()

In [29]:
def hybrid_recommendations(movie_name):
    content = content_recommendations(movie_name)
    collab = collaborative_recommendations(movie_name)

    combined = list(dict.fromkeys(content + collab))  

    return combined[:5]

In [30]:
def recommend_all(movie_name):
    print(f"\n🎬 Recommendations for: {movie_name}\n")

    print(" Popular:")
    print(popular_recommendations())

    print("\n Trending:")
    print(trending_recommendations())

    print("\n Content-Based:")
    print(content_recommendations(movie_name))

    print("\n Collaborative:")
    print(collaborative_recommendations(movie_name))

    print("\n Hybrid:")
    print(hybrid_recommendations(movie_name))

In [55]:
recommend_all("Star Wars (1977)")


🎬 Recommendations for: Star Wars (1977)

 Popular:
['Close Shave, A (1995)', "Schindler's List (1993)", 'Wrong Trousers, The (1993)', 'Casablanca (1942)', 'Wallace & Gromit: The Best of Aardman Animation (1996)']

 Trending:
       user_id  item_id  rating           timestamp  \
1          186      302       3 1998-04-04 19:22:22   
4          166      346       1 1998-02-02 05:33:16   
5          298      474       4 1998-01-07 14:20:06   
7          253      465       5 1998-04-03 18:34:27   
8          305      451       3 1998-02-01 09:20:17   
...        ...      ...     ...                 ...   
99985      617      582       4 1998-01-03 01:01:34   
99987      660      229       2 1998-04-01 04:50:12   
99988      421      498       4 1998-04-10 20:49:04   
99989      495     1091       4 1998-02-28 03:45:03   
99991      676      538       4 1998-04-16 00:10:37   

                                                   title  
1                               L.A. Confidential (199

## THE END!!

## 